<a href="https://colab.research.google.com/github/bhushan-madankar/AQI-Predictor/blob/main/Bhushan_Madankar_Cyber.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, lower, to_date, to_timestamp, when, sum as _sum, count, avg, month, year, date_format

# Initialize Spark Session
spark = SparkSession.builder.appName("EXL_Mini_Project").getOrCreate()
print("Spark Session Created Successfully")


Spark Session Created Successfully


In [ ]:
df_users = spark.read.csv("users.csv", header=True, inferSchema=True)
df_products = spark.read.csv("products.csv", header=True, inferSchema=True)
df_orders = spark.read.csv("orders.csv", header=True, inferSchema=True)
df_items = spark.read.csv("order_items.csv", header=True, inferSchema=True)

# Check Data (Run these one by one to inspect)
df_users.show(5)
df_users.printSchema()
print(f"Users Count: {df_users.count()}")


+-------+---------------+--------+
|user_id|           name|    city|
+-------+---------------+--------+
|   4930|     Amit gowda|Ludhiana|
|   2025|      Aman Bose|  Meerut|
|   2797|  Kavita sharma|  Jaipur|
|   2471|  Sophia taylor|  Jaipur|
|   2528|Anaya  Malhotra| Gurgaon|
+-------+---------------+--------+
only showing top 5 rows

root
 |-- user_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)

Users Count: 5250


In [5]:
df_users_clean = df_users.dropna(subset=["user_id"]) \
    .dropDuplicates(["user_id"]) \
    .withColumn("city", lower(trim(col("city")))) \
    .withColumn("name", trim(col("name")))

# Verify
df_users_clean.show(5)

+-------+-------------+---------+
|user_id|         name|     city|
+-------+-------------+---------+
|      1|         NULL|  jodhpur|
|      2|OLIVIA Sharma|     pune|
|      3|Anaya  kapoor|hyderabad|
|      4|         NULL|  gwalior|
|      5|         NULL| amritsar|
+-------+-------------+---------+
only showing top 5 rows



In [7]:
df_products_clean = df_products.dropna(subset=["product_id"]) \
    .dropDuplicates(["product_id"]) \
    .withColumn("category", lower(trim(col("category"))))

# Verify
df_products_clean.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- sku: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)



In [29]:
df_products_clean = df_products.dropna(subset=["product_id"]) \
    .dropDuplicates(["product_id"]) \
    .withColumn("category", lower(trim(col("category")))) \
    .withColumn("sku", lower(trim(col("sku"))))

# Verify
df_products_clean.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- sku: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)



In [31]:
df_items_clean = df_items.dropna(subset=["order_item_id"]) \
    .dropDuplicates(["order_item_id"]) \
    .withColumn("quantity", col("quantity").cast("integer")) \
    .withColumn("price", col("price").cast("float")) \
    .withColumn("sku", lower(trim(col("sku"))))

In [32]:
from pyspark.sql.functions import to_timestamp, to_date

df_orders_clean = df_orders.dropna(subset=["order_id"]) \
    .dropDuplicates(["order_id"]) \
    .withColumn("amount", col("amount").cast("float")) \
    .withColumn("user_id", col("user_id").cast("integer")) \
    .withColumn("order_timestamp", to_timestamp(col("order_timestamp"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("order_date", to_date(col("order_timestamp")))

# Verify
df_orders_clean.printSchema()
df_orders_clean.show(5)

root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)
 |-- amount: float (nullable = true)
 |-- order_date: date (nullable = true)

+--------+-------+---------------+------+----------+
|order_id|user_id|order_timestamp|amount|order_date|
+--------+-------+---------------+------+----------+
|       1|   NULL|           NULL|1500.0|      NULL|
|       2|   1036|           NULL|3129.0|      NULL|
|       3|   2258|           NULL|2773.0|      NULL|
|       4|   1852|           NULL| 500.5|      NULL|
|       5|   2839|           NULL|3682.0|      NULL|
+--------+-------+---------------+------+----------+
only showing top 5 rows



In [33]:
unique_dates = df_orders_clean.select("order_date").distinct()

from pyspark.sql.functions import dayofmonth, month, year, quarter, dayofweek

dim_date = unique_dates.withColumn("date_key", col("order_date")) \
    .withColumn("year", year("order_date")) \
    .withColumn("month", month("order_date")) \
    .withColumn("day", dayofmonth("order_date")) \
    .withColumn("quarter", quarter("order_date")) \
    .withColumn("day_of_week", dayofweek("order_date"))

dim_date.createOrReplaceTempView("dim_date")


In [34]:
df_orders_clean.createOrReplaceTempView("fact_orders")
df_users_clean.createOrReplaceTempView("dim_users")

spark.sql("""
    SELECT f.order_id, f.amount as sales_amount, u.name as user_name
    FROM fact_orders f
    JOIN dim_users u ON f.user_id = u.user_id
    WHERE lower(u.city) = 'mumbai'
    AND MONTH(f.order_date) = 12
""").show()

+--------+------------+--------------+
|order_id|sales_amount|     user_name|
+--------+------------+--------------+
|    9578|       220.0|   Aman  Joshi|
|   13451|       500.5|  yash Chauhan|
|   17802|      1959.0| Gaurav  Dutta|
|   18600|      3497.0|john Fernandes|
+--------+------------+--------------+



In [35]:
spark.sql("""
    SELECT
        u.city,
        SUM(f.amount) as city_sales,
        (SUM(f.amount) / (SELECT SUM(amount) FROM fact_orders)) * 100 as percentage
    FROM fact_orders f
    JOIN dim_users u ON f.user_id = u.user_id
    GROUP BY u.city
""").show()

+-------------+----------+------------------+
|         city|city_sales|        percentage|
+-------------+----------+------------------+
|      chennai|  256846.0|1.0905437114856613|
|       nashik|  255568.5|1.0851195678687744|
|        thane|  208023.0|0.8832458924584449|
|        delhi|  437278.0|1.8566408395343008|
|     amritsar|  202888.0|0.8614431703663005|
|     thrissur|  260741.0|1.1070815113978136|
|       meerut|  228536.0|0.9703421413924572|
|        patna|  290035.5|1.2314631749476321|
|    allahabad|  266896.5|1.1332171795259913|
|      lucknow|  192781.0|0.8185298086943821|
|visakhapatnam|  248685.5| 1.055895003864835|
|         NULL|  263365.5|1.1182248890279658|
|       howrah|  177603.0|0.7540854628492868|
|       bhopal|  267432.5|1.1354929845973427|
|        surat|  256193.0|1.0877711355311979|
|        salem|  164757.0|0.6995425674265633|
|       ranchi|  240852.5|1.0226368301262243|
|      jodhpur|  282514.5|1.1995297235639872|
|    ahmedabad|  258530.0| 1.09769

In [36]:
spark.sql("""
    SELECT u.name as user_name, AVG(total_order_val) as avg_order_value
    FROM (
        SELECT user_id, order_id, SUM(amount) as total_order_val
        FROM fact_orders
        GROUP BY user_id, order_id
    ) sub
    JOIN dim_users u ON sub.user_id = u.user_id
    GROUP BY u.name
""").show()

+----------------+------------------+
|       user_name|   avg_order_value|
+----------------+------------------+
|    Amit  Ansari|             755.0|
|Siddharth mishra|           2202.25|
|     Kabir reddy|           1764.75|
|    Rahul Khanna|            1500.0|
|   Kabir Johnson|            565.75|
|     Kavita Khan|           1000.25|
|     Rhea  Yadav|            974.75|
|   Anita  Sheikh|              NULL|
|     Maria  Bose|            1500.0|
|    Vihaan dutta|              NULL|
|Deepak fernandes|            1617.5|
|    Vihaan Gupta|             500.5|
|      Sophia Roy|2313.8333333333335|
|  Farhan  Tiwari|1452.6666666666667|
|    Arjun Thakur|            3178.5|
|     Noah  Clark|            1500.0|
|   Rahul Chauhan|              NULL|
|     komal Brown|            4984.0|
|      kabir Bose|            2038.7|
|    Arjun Mishra|1388.6666666666667|
+----------------+------------------+
only showing top 20 rows



In [37]:
spark.sql("""
    SELECT u.name as user_name, COUNT(DISTINCT f.order_id) as order_count
    FROM fact_orders f
    JOIN dim_users u ON f.user_id = u.user_id
    GROUP BY u.name
    HAVING COUNT(DISTINCT f.order_id) > 1
    ORDER BY order_count DESC
""").show()

+---------------+-----------+
|      user_name|order_count|
+---------------+-----------+
|           NULL|       1629|
| Emma Fernandes|         18|
|  Sophia Thakur|         16|
|    James Yadav|         14|
|     Rahul bose|         14|
|     Komal Bose|         14|
|   Aman Agarwal|         14|
|     emma Brown|         13|
|   Kabir Mishra|         13|
|    Neha Mishra|         13|
|   Sneha sheikh|         13|
|   Sophia Verma|         13|
|    Anita Gowda|         12|
| Arjun Anderson|         12|
|    arjun Patel|         12|
|   Pooja Khanna|         12|
|     Rani Brown|         12|
|     Asha Naidu|         12|
|kavita Anderson|         11|
|    Ishaan Bose|         11|
+---------------+-----------+
only showing top 20 rows



In [38]:
spark.sql("""
    SELECT YEAR(order_date) as year, MONTH(order_date) as month, SUM(amount) as revenue
    FROM fact_orders
    GROUP BY YEAR(order_date), MONTH(order_date)
    ORDER BY year, month
""").show()

+----+-----+-----------+
|year|month|    revenue|
+----+-----+-----------+
|NULL| NULL|1.9618453E7|
|2022|    1|    97543.5|
|2022|    2|    61014.0|
|2022|    3|   151658.0|
|2022|    4|   103680.0|
|2022|    5|   101198.5|
|2022|    6|   101311.0|
|2022|    7|   100945.0|
|2022|    8|   132074.5|
|2022|    9|   120610.5|
|2022|   10|   105049.0|
|2022|   11|   113569.0|
|2022|   12|   115723.5|
|2023|    1|   123511.5|
|2023|    2|    84858.0|
|2023|    3|    88699.0|
|2023|    4|   135301.5|
|2023|    5|   131060.0|
|2023|    6|   116890.5|
|2023|    7|   106138.5|
+----+-----+-----------+
only showing top 20 rows



In [39]:
df_products_clean.createOrReplaceTempView("dim_products")
df_items_clean.createOrReplaceTempView("fact_order_items")

spark.sql("""
    SELECT p.product_name, SUM(fi.price * fi.quantity) as total_revenue
    FROM fact_order_items fi
    JOIN dim_products p ON fi.sku = p.sku
    GROUP BY p.product_name
    ORDER BY total_revenue DESC
    LIMIT 5
""").show()

+------------+-------------+
|product_name|total_revenue|
+------------+-------------+
|FICTION BOOK|         NULL|
+------------+-------------+



In [40]:
def check_quality(df, name):
    print(f"--- {name} ---")
    print(f"Total Rows: {df.count()}")
    print(f"Duplicates: {df.count() - df.dropDuplicates().count()}")
